# PERSUADE: Local Lag Curve Analysis (Sliding Windows)

**Goal:** Measure short-range context sensitivity by varying how much immediately preceding context the model gets.

**Method:**
- Sample multiple target windows (T=32 tokens) throughout each essay
- For each window, vary context length k from 0 to 128 tokens
- Compute NLL(k) and gain_k = NLL(0) - NLL(k)
- Aggregate within essay, then compare curve shapes across groups

**Why sliding windows?**
- Using only final 256 tokens overweights conclusion-style language
- Sliding windows estimate typical context dependence throughout the essay
- Better within-essay reliability for individual differences

**Cohort:** persuade_score_long_cohort.jsonl (50/50/50 balanced, all long enough)

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes pandas numpy matplotlib seaborn tqdm statsmodels

In [ ]:
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
import statsmodels.formula.api as smf

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Paths
DRIVE_BASE = "/content/drive/MyDrive/LRTIA/Data/persuade_clean"
OUTPUT_BASE = "/content/drive/MyDrive/LRTIA/Results/Persuade"
COHORT_PATH = f"{DRIVE_BASE}/cohorts/persuade_score_long_cohort.jsonl"

EXPERIMENT = 'local_lag_curve_sliding_v1'

# Model
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True  # True for T4, False for A100

# Sliding window parameters
TARGET_LENGTH = 32  # T=32 tokens per window (sensitive to immediate context)
N_WINDOWS = 20      # Number of windows to sample per essay
MIN_CONTEXT = 128   # Minimum tokens of preceding context required
END_BUFFER = 64     # Exclude last 64 tokens to avoid "ending" effects

# Context length grid (k values)
K_GRID = [0, 2, 4, 8, 12, 16, 24, 32, 48, 64, 96, 128]

# Minimum tokens required: context (128) + target (32) + end buffer (64)
MIN_TOKENS_REQUIRED = MIN_CONTEXT + TARGET_LENGTH + END_BUFFER

# Reproducibility
RANDOM_SEED = 42

print(f"Experiment: {EXPERIMENT}")
print(f"Cohort: {COHORT_PATH}")
print(f"\nSliding window parameters:")
print(f"  Target length T: {TARGET_LENGTH} tokens")
print(f"  Windows per essay: {N_WINDOWS}")
print(f"  Min preceding context: {MIN_CONTEXT} tokens")
print(f"  End buffer (excluded): {END_BUFFER} tokens")
print(f"\nContext grid k: {K_GRID}")
print(f"Min tokens required: {MIN_TOKENS_REQUIRED}")
print(f"Random seed: {RANDOM_SEED}")

## 1. Load Model and Data

In [ ]:
# Load tokenizer
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load model
print(f"Loading model: {MODEL_NAME}")

if USE_4BIT:
    print("  Using 4-bit quantization")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
else:
    print("  Using float16")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )

model.eval()
print("Model loaded")

In [ ]:
# Load cohort
def load_cohort(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

cohort = load_cohort(COHORT_PATH)
print(f"Loaded {len(cohort)} essays")

# Verify cohort
df_cohort = pd.DataFrame(cohort)
print(f"\nScore bin distribution:")
print(df_cohort['score_bin'].value_counts())

In [ ]:
# Pre-tokenize and filter
essay_tokens = {}
excluded = 0

for essay in cohort:
    token_ids = tokenizer.encode(essay['text'], add_special_tokens=False)
    if len(token_ids) >= MIN_TOKENS_REQUIRED:
        essay_tokens[essay['essay_id']] = token_ids
    else:
        excluded += 1

print(f"Tokenized essays: {len(essay_tokens)}")
print(f"Excluded (too short): {excluded}")

lengths = [len(t) for t in essay_tokens.values()]
print(f"Token lengths: min={min(lengths)}, median={np.median(lengths):.0f}, max={max(lengths)}")

## 2. Core Functions

In [ ]:
@torch.no_grad()
def compute_nll_for_target(context_ids, target_ids):
    """
    Compute mean NLL for predicting target_ids given context_ids.
    
    Args:
        context_ids: list of token ids for context (can be empty)
        target_ids: list of token ids for target region
    
    Returns:
        mean NLL over target tokens
    """
    full_ids = context_ids + target_ids
    
    if len(context_ids) == 0:
        # No context - predict within target only
        input_ids = torch.tensor([full_ids], device=model.device)
        outputs = model(input_ids)
        logits = outputs.logits[0]
        
        nlls = []
        for i in range(len(full_ids) - 1):
            log_probs = torch.log_softmax(logits[i], dim=-1)
            true_token = full_ids[i + 1]
            nll = -log_probs[true_token].item()
            nlls.append(nll)
        
        return np.mean(nlls) if nlls else 0.0
    else:
        # With context: predict target tokens
        input_ids = torch.tensor([full_ids], device=model.device)
        outputs = model(input_ids)
        logits = outputs.logits[0]
        
        target_start = len(context_ids)
        nlls = []
        
        # Prediction of first target token from last context token
        log_probs = torch.log_softmax(logits[target_start - 1], dim=-1)
        nll_first = -log_probs[target_ids[0]].item()
        nlls.append(nll_first)
        
        # Predictions within target
        for i in range(len(target_ids) - 1):
            pos = target_start + i
            if pos >= logits.shape[0]:
                break
            log_probs = torch.log_softmax(logits[pos], dim=-1)
            true_token = target_ids[i + 1]
            nll = -log_probs[true_token].item()
            nlls.append(nll)
        
        return np.mean(nlls) if nlls else 0.0


def sample_window_positions(n_tokens, n_windows, rng):
    """
    Sample target window start positions.
    
    Valid range: [MIN_CONTEXT, n_tokens - TARGET_LENGTH - END_BUFFER)
    """
    min_start = MIN_CONTEXT
    max_start = n_tokens - TARGET_LENGTH - END_BUFFER
    
    if max_start <= min_start:
        return []
    
    # Uniformly spaced positions
    if n_windows == 1:
        return [(min_start + max_start) // 2]
    
    # Create evenly spaced positions
    positions = np.linspace(min_start, max_start, n_windows, dtype=int).tolist()
    
    # Add small random jitter for variety (within bounds)
    jittered = []
    for pos in positions:
        jitter = rng.randint(-5, 5)
        new_pos = max(min_start, min(max_start, pos + jitter))
        jittered.append(new_pos)
    
    return jittered


def run_lag_curve_for_window(token_ids, target_start):
    """
    Compute NLL for each k for a single target window.
    
    Returns dict with k -> nll and k -> gain.
    """
    target_end = target_start + TARGET_LENGTH
    target_ids = token_ids[target_start:target_end]
    context_pool = token_ids[:target_start]
    
    nll_by_k = {}
    
    for k in K_GRID:
        if k == 0:
            context_ids = []
        else:
            context_ids = context_pool[-k:] if k <= len(context_pool) else context_pool
        
        nll = compute_nll_for_target(context_ids, target_ids)
        nll_by_k[k] = nll
    
    # Compute gains
    nll_0 = nll_by_k[0]
    gain_by_k = {k: nll_0 - nll_by_k[k] for k in K_GRID}
    
    return {'nll': nll_by_k, 'gain': gain_by_k}


def run_sliding_lag_curve_for_essay(token_ids, essay_id, rng):
    """
    Run sliding window lag curve analysis for a single essay.
    
    Returns:
        - essay_results: dict with aggregated metrics
        - window_results: list of per-window results
    """
    n_tokens = len(token_ids)
    
    # Sample window positions
    positions = sample_window_positions(n_tokens, N_WINDOWS, rng)
    
    if len(positions) == 0:
        return None, []
    
    # Collect results for each window
    window_results = []
    all_gains = {k: [] for k in K_GRID}
    all_nlls = {k: [] for k in K_GRID}
    
    for win_idx, target_start in enumerate(positions):
        win_result = run_lag_curve_for_window(token_ids, target_start)
        
        # Store window-level data
        for k in K_GRID:
            window_results.append({
                'essay_id': essay_id,
                'window_idx': win_idx,
                'target_start': target_start,
                'k': k,
                'nll_k': win_result['nll'][k],
                'gain_k': win_result['gain'][k],
            })
            all_gains[k].append(win_result['gain'][k])
            all_nlls[k].append(win_result['nll'][k])
    
    # Aggregate across windows
    essay_results = {'essay_id': essay_id, 'n_windows': len(positions)}
    
    # Mean NLL and gain per k
    for k in K_GRID:
        essay_results[f'mean_nll_{k}'] = np.mean(all_nlls[k])
        essay_results[f'mean_gain_{k}'] = np.mean(all_gains[k])
        essay_results[f'std_gain_{k}'] = np.std(all_gains[k])
    
    # Shape metrics from aggregated curve
    mean_gain_16 = essay_results['mean_gain_16']
    mean_gain_128 = essay_results['mean_gain_128']
    
    # early_ratio
    if mean_gain_128 > 0:
        essay_results['early_ratio'] = mean_gain_16 / mean_gain_128
    else:
        essay_results['early_ratio'] = np.nan
    
    # log_slope_local: fit line to (log(k+1), mean_gain_k) for k >= 2
    ks_for_fit = [k for k in K_GRID if k >= 2]
    log_ks = [np.log(k + 1) for k in ks_for_fit]
    gains_for_fit = [essay_results[f'mean_gain_{k}'] for k in ks_for_fit]
    
    if len(log_ks) >= 2:
        slope, intercept, r_value, p_value, std_err = stats.linregress(log_ks, gains_for_fit)
        essay_results['log_slope_local'] = slope
        essay_results['log_slope_r2'] = r_value ** 2
    else:
        essay_results['log_slope_local'] = np.nan
        essay_results['log_slope_r2'] = np.nan
    
    # Stability metrics (within-essay variance)
    essay_results['stability_gain_128'] = np.std(all_gains[128])
    
    # Per-window early_ratio variance
    window_early_ratios = []
    for i in range(len(positions)):
        g16 = all_gains[16][i]
        g128 = all_gains[128][i]
        if g128 > 0:
            window_early_ratios.append(g16 / g128)
    if window_early_ratios:
        essay_results['stability_early_ratio'] = np.std(window_early_ratios)
    else:
        essay_results['stability_early_ratio'] = np.nan
    
    # Baseline NLL (mean NLL at k=128 across windows)
    essay_results['baseline_nll'] = essay_results['mean_nll_128']
    
    return essay_results, window_results


print("Core functions defined")

## 3. Run Sliding Window Analysis

In [ ]:
# Build essay metadata lookup
essay_meta = {e['essay_id']: e for e in cohort}

# Initialize RNG
rng = random.Random(RANDOM_SEED)

# Process all essays
all_essay_results = []
all_window_results = []
start_time = time.time()

for essay_id, token_ids in tqdm(essay_tokens.items(), desc="Processing essays"):
    essay_results, window_results = run_sliding_lag_curve_for_essay(token_ids, essay_id, rng)
    
    if essay_results is None:
        continue
    
    # Add metadata
    meta = essay_meta[essay_id]
    essay_results['score'] = meta.get('score')
    essay_results['score_bin'] = meta.get('score_bin')
    essay_results['grade'] = meta.get('grade')
    essay_results['token_count'] = len(token_ids)
    
    all_essay_results.append(essay_results)
    all_window_results.extend(window_results)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f}s ({elapsed/len(essay_tokens):.2f}s/essay)")
print(f"Essays processed: {len(all_essay_results)}")
print(f"Total windows: {len(all_window_results) // len(K_GRID)}")

In [ ]:
# Create DataFrames
df = pd.DataFrame(all_essay_results)
df_windows = pd.DataFrame(all_window_results)

print(f"Essay-level results: {df.shape}")
print(f"Window-level results: {df_windows.shape}")

df.head()

## 4. Sanity Checks

In [ ]:
print("="*80)
print("SANITY CHECK: Mean Gain by k (should increase with k)")
print("="*80)

for k in K_GRID:
    col = f'mean_gain_{k}'
    print(f"  k={k:3d}: mean={df[col].mean():.4f}, std={df[col].std():.4f}")

In [ ]:
print("\n" + "="*80)
print("SANITY CHECK: Shape Metrics")
print("="*80)

print(f"\nearly_ratio (mean_gain_16 / mean_gain_128):")
print(f"  mean={df['early_ratio'].mean():.4f}, median={df['early_ratio'].median():.4f}")
print(f"  range=[{df['early_ratio'].min():.4f}, {df['early_ratio'].max():.4f}]")
print(f"  NaN count: {df['early_ratio'].isna().sum()}")

print(f"\nlog_slope_local:")
print(f"  mean={df['log_slope_local'].mean():.4f}, median={df['log_slope_local'].median():.4f}")
print(f"  range=[{df['log_slope_local'].min():.4f}, {df['log_slope_local'].max():.4f}]")

print(f"\nmean_gain_128 (total local benefit):")
print(f"  mean={df['mean_gain_128'].mean():.4f}, median={df['mean_gain_128'].median():.4f}")

In [ ]:
print("\n" + "="*80)
print("SANITY CHECK: Within-Essay Stability")
print("="*80)

print(f"\nstability_gain_128 (SD of gain_128 across windows):")
print(f"  mean={df['stability_gain_128'].mean():.4f}")

print(f"\nstability_early_ratio (SD of early_ratio across windows):")
print(f"  mean={df['stability_early_ratio'].mean():.4f}")

# Check that within-essay SD is smaller than between-essay SD
between_essay_sd = df['early_ratio'].std()
mean_within_essay_sd = df['stability_early_ratio'].mean()
print(f"\nBetween-essay SD of early_ratio: {between_essay_sd:.4f}")
print(f"Mean within-essay SD of early_ratio: {mean_within_essay_sd:.4f}")
print(f"Ratio (between/within): {between_essay_sd/mean_within_essay_sd:.2f}x")

## 5. Group Analysis

In [ ]:
GROUP_ORDER = ['low', 'mid', 'high']
SCORE_COLORS = {'low': '#e74c3c', 'mid': '#f39c12', 'high': '#2ecc71'}

df['score_bin'] = pd.Categorical(df['score_bin'], categories=GROUP_ORDER, ordered=True)

In [ ]:
print("="*80)
print("GROUP MEANS: Shape Metrics by Score Bin")
print("="*80)

for metric in ['early_ratio', 'log_slope_local', 'mean_gain_128', 'baseline_nll', 'stability_early_ratio']:
    print(f"\n{metric}:")
    for score_bin in GROUP_ORDER:
        subset = df[df['score_bin'] == score_bin][metric].dropna()
        mean = subset.mean()
        sem = subset.sem()
        print(f"  {score_bin}: {mean:.4f} +/- {1.96*sem:.4f} (n={len(subset)})")

In [ ]:
# Build gain curve summary by group
curve_rows = []

for score_bin in GROUP_ORDER:
    subset = df[df['score_bin'] == score_bin]
    for k in K_GRID:
        gain_col = f'mean_gain_{k}'
        nll_col = f'mean_nll_{k}'
        
        curve_rows.append({
            'score_bin': score_bin,
            'k': k,
            'gain_mean': subset[gain_col].mean(),
            'gain_sem': subset[gain_col].sem(),
            'gain_ci95': 1.96 * subset[gain_col].sem(),
            'nll_mean': subset[nll_col].mean(),
            'n': len(subset),
        })

df_curve = pd.DataFrame(curve_rows)
print("Group curve summary (mean_gain_k):")
print(df_curve.pivot(index='k', columns='score_bin', values='gain_mean').round(4))

## 6. Statistical Tests

In [ ]:
# Standardize controls
df['token_count_z'] = (df['token_count'] - df['token_count'].mean()) / df['token_count'].std()
df['baseline_nll_z'] = (df['baseline_nll'] - df['baseline_nll'].mean()) / df['baseline_nll'].std()

# Drop rows with NaN in key metrics
df_valid = df.dropna(subset=['early_ratio', 'log_slope_local']).copy()
print(f"Valid essays for regression: {len(df_valid)} / {len(df)}")

In [ ]:
print("="*80)
print("REGRESSION: early_ratio ~ C(score_bin) + controls")
print("="*80)

formula_early = 'early_ratio ~ C(score_bin) + token_count_z + baseline_nll_z'
model_early = smf.ols(formula_early, data=df_valid).fit()

print(f"\nFormula: {formula_early}")
print(f"R²: {model_early.rsquared:.4f}, Adj R²: {model_early.rsquared_adj:.4f}, n={int(model_early.nobs)}")

print("\nCoefficients:")
for param in model_early.params.index:
    coef = model_early.params[param]
    pval = model_early.pvalues[param]
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

In [ ]:
print("\n" + "="*80)
print("REGRESSION: log_slope_local ~ C(score_bin) + controls")
print("="*80)

formula_slope = 'log_slope_local ~ C(score_bin) + token_count_z + baseline_nll_z'
model_slope = smf.ols(formula_slope, data=df_valid).fit()

print(f"\nFormula: {formula_slope}")
print(f"R²: {model_slope.rsquared:.4f}, Adj R²: {model_slope.rsquared_adj:.4f}, n={int(model_slope.nobs)}")

print("\nCoefficients:")
for param in model_slope.params.index:
    coef = model_slope.params[param]
    pval = model_slope.pvalues[param]
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

In [ ]:
print("\n" + "="*80)
print("REGRESSION: mean_gain_128 ~ C(score_bin) + controls")
print("="*80)

formula_gain = 'mean_gain_128 ~ C(score_bin) + token_count_z + baseline_nll_z'
model_gain = smf.ols(formula_gain, data=df_valid).fit()

print(f"\nFormula: {formula_gain}")
print(f"R²: {model_gain.rsquared:.4f}, Adj R²: {model_gain.rsquared_adj:.4f}, n={int(model_gain.nobs)}")

print("\nCoefficients:")
for param in model_gain.params.index:
    coef = model_gain.params[param]
    pval = model_gain.pvalues[param]
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

## 7. Visualizations

In [ ]:
# Create output directory
output_dir = Path(OUTPUT_BASE) / EXPERIMENT
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

In [ ]:
# PLOT 1: Group mean gain curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Linear x-axis
ax = axes[0]
for score_bin in GROUP_ORDER:
    subset = df_curve[df_curve['score_bin'] == score_bin]
    ax.errorbar(subset['k'], subset['gain_mean'], yerr=subset['gain_ci95'],
                marker='o', capsize=3, label=score_bin, color=SCORE_COLORS[score_bin],
                linewidth=2, markersize=6)

ax.set_xlabel('Context length k (tokens)', fontsize=11)
ax.set_ylabel('Mean Gain = NLL(0) - NLL(k)', fontsize=11)
ax.set_title('Gain Curve by Score Bin\n(linear scale, averaged over windows)', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

# Right: Log x-axis
ax = axes[1]
for score_bin in GROUP_ORDER:
    subset = df_curve[df_curve['score_bin'] == score_bin]
    subset_nonzero = subset[subset['k'] > 0]
    ax.errorbar(subset_nonzero['k'], subset_nonzero['gain_mean'], yerr=subset_nonzero['gain_ci95'],
                marker='o', capsize=3, label=score_bin, color=SCORE_COLORS[score_bin],
                linewidth=2, markersize=6)

ax.set_xscale('log')
ax.set_xlabel('Context length k (tokens, log scale)', fontsize=11)
ax.set_ylabel('Mean Gain = NLL(0) - NLL(k)', fontsize=11)
ax.set_title('Gain Curve by Score Bin\n(log scale)', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig(output_dir / 'gain_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PLOT 2: Distribution of early_ratio by group
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot
ax = axes[0]
data_for_plot = [df[df['score_bin'] == sb]['early_ratio'].dropna() for sb in GROUP_ORDER]
parts = ax.violinplot(data_for_plot, positions=range(len(GROUP_ORDER)), showmeans=True, showmedians=True)

for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(SCORE_COLORS[GROUP_ORDER[i]])
    pc.set_alpha(0.7)

ax.set_xticks(range(len(GROUP_ORDER)))
ax.set_xticklabels(GROUP_ORDER)
ax.set_xlabel('Score Bin', fontsize=11)
ax.set_ylabel('early_ratio (gain_16 / gain_128)', fontsize=11)
ax.set_title('Distribution of Early Saturation Ratio\n(higher = faster saturation)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Box plot
ax = axes[1]
df_valid.boxplot(column='early_ratio', by='score_bin', ax=ax,
                  positions=range(len(GROUP_ORDER)))
ax.set_xlabel('Score Bin', fontsize=11)
ax.set_ylabel('early_ratio', fontsize=11)
ax.set_title('early_ratio by Score Bin', fontsize=12, fontweight='bold')
plt.suptitle('')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(output_dir / 'early_ratio_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PLOT 3: Scatter early_ratio vs baseline_nll (independence check)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for score_bin in GROUP_ORDER:
    subset = df_valid[df_valid['score_bin'] == score_bin]
    ax.scatter(subset['baseline_nll'], subset['early_ratio'], 
               alpha=0.5, label=score_bin, color=SCORE_COLORS[score_bin], s=30)

ax.set_xlabel('Baseline NLL (fluency proxy)', fontsize=11)
ax.set_ylabel('early_ratio', fontsize=11)
ax.set_title('early_ratio vs Fluency\n(checking independence)', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

r, p = stats.pearsonr(df_valid['baseline_nll'], df_valid['early_ratio'])
ax.text(0.05, 0.95, f'r = {r:.3f}, p = {p:.4f}', transform=ax.transAxes, 
        fontsize=10, verticalalignment='top')

ax = axes[1]
for score_bin in GROUP_ORDER:
    subset = df_valid[df_valid['score_bin'] == score_bin]
    ax.scatter(subset['baseline_nll'], subset['log_slope_local'], 
               alpha=0.5, label=score_bin, color=SCORE_COLORS[score_bin], s=30)

ax.set_xlabel('Baseline NLL (fluency proxy)', fontsize=11)
ax.set_ylabel('log_slope_local', fontsize=11)
ax.set_title('Log Slope vs Fluency\n(checking independence)', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

r, p = stats.pearsonr(df_valid['baseline_nll'], df_valid['log_slope_local'])
ax.text(0.05, 0.95, f'r = {r:.3f}, p = {p:.4f}', transform=ax.transAxes, 
        fontsize=10, verticalalignment='top')

plt.tight_layout()
plt.savefig(output_dir / 'shape_vs_fluency.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PLOT 4: Reliability - within-essay stability by group
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stability of gain_128
ax = axes[0]
data = [df[df['score_bin'] == sb]['stability_gain_128'].dropna() for sb in GROUP_ORDER]
parts = ax.violinplot(data, positions=range(len(GROUP_ORDER)), showmeans=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(SCORE_COLORS[GROUP_ORDER[i]])
    pc.set_alpha(0.7)

ax.set_xticks(range(len(GROUP_ORDER)))
ax.set_xticklabels(GROUP_ORDER)
ax.set_xlabel('Score Bin', fontsize=11)
ax.set_ylabel('Within-essay SD of gain_128', fontsize=11)
ax.set_title('Stability of gain_128 Across Windows\n(lower = more consistent)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Stability of early_ratio
ax = axes[1]
data = [df[df['score_bin'] == sb]['stability_early_ratio'].dropna() for sb in GROUP_ORDER]
parts = ax.violinplot(data, positions=range(len(GROUP_ORDER)), showmeans=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(SCORE_COLORS[GROUP_ORDER[i]])
    pc.set_alpha(0.7)

ax.set_xticks(range(len(GROUP_ORDER)))
ax.set_xticklabels(GROUP_ORDER)
ax.set_xlabel('Score Bin', fontsize=11)
ax.set_ylabel('Within-essay SD of early_ratio', fontsize=11)
ax.set_title('Stability of early_ratio Across Windows\n(lower = more consistent)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(output_dir / 'reliability_by_group.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PLOT 5: log_slope_local distribution
fig, ax = plt.subplots(figsize=(10, 5))

for score_bin in GROUP_ORDER:
    subset = df_valid[df_valid['score_bin'] == score_bin]['log_slope_local']
    ax.hist(subset, bins=25, alpha=0.5, label=score_bin, color=SCORE_COLORS[score_bin], density=True)

ax.set_xlabel('log_slope_local', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Distribution of Log Slope (Saturation Rate)\n(higher = steeper saturation)', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'log_slope_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Results

In [ ]:
# Save essay-level results
df.to_csv(output_dir / 'essay_level_results_local_lag.csv', index=False)

# Save window-level results
df_windows.to_csv(output_dir / 'window_level_results_local_lag.csv', index=False)

# Save group curve summary
df_curve.to_csv(output_dir / 'group_curve_summary.csv', index=False)

# Save regression results
with open(output_dir / 'regression_local_lag.txt', 'w') as f:
    f.write("LOCAL LAG CURVE REGRESSION ANALYSIS (SLIDING WINDOWS)\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Configuration:\n")
    f.write(f"  Cohort: persuade_score_long (50/50/50)\n")
    f.write(f"  Target length T: {TARGET_LENGTH} tokens\n")
    f.write(f"  Windows per essay: {N_WINDOWS}\n")
    f.write(f"  Min preceding context: {MIN_CONTEXT} tokens\n")
    f.write(f"  End buffer: {END_BUFFER} tokens\n")
    f.write(f"  Context grid k: {K_GRID}\n")
    f.write(f"  Essays analyzed: {len(df)}\n")
    f.write(f"  Random seed: {RANDOM_SEED}\n")
    f.write("\n" + "=" * 80 + "\n\n")
    f.write("EARLY_RATIO MODEL:\n")
    f.write(f"Formula: {formula_early}\n")
    f.write(model_early.summary().as_text())
    f.write("\n\n" + "=" * 80 + "\n\n")
    f.write("LOG_SLOPE_LOCAL MODEL:\n")
    f.write(f"Formula: {formula_slope}\n")
    f.write(model_slope.summary().as_text())
    f.write("\n\n" + "=" * 80 + "\n\n")
    f.write("MEAN_GAIN_128 MODEL:\n")
    f.write(f"Formula: {formula_gain}\n")
    f.write(model_gain.summary().as_text())

print(f"\nSaved to {output_dir}/")
print(f"  - essay_level_results_local_lag.csv ({len(df)} rows)")
print(f"  - window_level_results_local_lag.csv ({len(df_windows)} rows)")
print(f"  - group_curve_summary.csv")
print(f"  - regression_local_lag.txt")
print(f"  - gain_curves.png")
print(f"  - early_ratio_distribution.png")
print(f"  - shape_vs_fluency.png")
print(f"  - reliability_by_group.png")
print(f"  - log_slope_distribution.png")

In [ ]:
# Final summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print(f"""
GOAL: Measure short-range context sensitivity via sliding-window lag curves.

METHOD:
  - Sample {N_WINDOWS} target windows (T={TARGET_LENGTH} tokens) per essay
  - For each window, compute NLL with varying context (k=0 to {max(K_GRID)})
  - Average gain_k across windows within each essay
  - Extract shape metrics from averaged curve

SHAPE METRICS:
  * early_ratio = mean_gain_16 / mean_gain_128
    (fraction of benefit from first 16 tokens)
  * log_slope_local = slope of gain vs log(k)
    (saturation rate)
  * stability metrics = within-essay SD
    (consistency across windows)

KEY QUESTIONS:
  1. Do groups differ in curve shape (not just fluency)?
  2. Is early_ratio different across score bins?
  3. Are shape metrics independent of baseline fluency?
  4. Is within-essay reliability different by group?

INTERPRETATION:
  - Higher early_ratio = faster saturation (first 16 tokens capture most benefit)
  - Higher log_slope_local = steeper gain curve
  - Lower stability = less consistent context dependence across essay
""")